# Process a device and post the results

The cycle that Smart Citizen Flows runs every few hours for each device with hardware: load the data since the last processing, process it with the blueprint of its hardware, check it, post the calculated channels to the Smart Citizen API, and save the date of the last processing.

Posting needs a token of the owner of the device (`SC_BEARER`). Here every post is a `dry_run`: set it to `False` to post.

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

In [ ]:
device = sc.Device(params=sc.APIParams(id=18154))
device.valid_for_processing

The postprocessing information of the device in the platform: its hardware, and the date of the last processing (`latest_postprocessing`).

In [ ]:
postprocessing = device.handler.json.postprocessing
print(device.blueprint)
print(postprocessing.hardware_url)
print(postprocessing.latest_postprocessing)

## Load

From the last processing, at most `limit` readings (flows loads 500 per run):

In [ ]:
import pandas as pd

latest = postprocessing.latest_postprocessing
device.options.min_date = (pd.Timestamp(latest) if latest else pd.Timestamp.now(tz='UTC') - pd.Timedelta('1D')).isoformat()
device.options.limit = 500
await device.load()
device.data.index.min(), device.data.index.max(), len(device.data)

## Process and check

In [ ]:
device.process()

In [ ]:
device.health_checks()
[(check['name'], check['status']) for check in device.health['checks']]

## Post

`columns='channels'` posts the channels of the blueprint marked with `post` (with the ids of the platform):

In [ ]:
[(channel.name, channel.id) for channel in device.channels if channel.post]

In [ ]:
await device.post(columns='channels', dry_run=True)

## Save the date of the last processing

`update_postprocessing_date` sets it to the last reading processed, and `update_postprocessing` saves it in the platform. The next run starts from there.

In [ ]:
device.update_postprocessing_date()
device.handler.latest_postprocessing

In [ ]:
device.update_postprocessing(dry_run=True)

`device.post(columns='channels', with_postprocessing=True)` posts the channels and saves the date in one call.